# 09 — Advanced: Dynamic Data Ingestion

> **DataMindAI | Ahmed**

## The real-world problem
You receive **hundreds of different assessment files** each night.
You need ONE ingestion notebook that handles all of them — driven by a mapping table.

## Architecture
```
config.file_mapping (SQL table)
    │
    ▼
generate_file_list  (reads config table, publishes array)
    │
    ▼
[For-Each: ingest_each_file]    (Inputs: {{tasks.generate_file_list.values.file_list}})
    │
    ├── Iteration 1: ds_assessment_jan.csv   → bronze.ds_raw
    ├── Iteration 2: eng_assessment_jan.csv  → bronze.eng_raw
    └── Iteration 3: maths_assessment_jan.csv → bronze.maths_raw
```

## Key principle
**Never hardcode file names.** Query the mapping table.
Adding a new file = one INSERT into `config.file_mapping`.  
Zero notebook code changes required.

---
## Mapping table setup
---

In [ ]:
# ── Create the file mapping table ────────────────────────────────────────────
# This is the config table that drives the entire pipeline.
# In production, this would be maintained by the data ops team.
from pyspark.sql import Row

file_mapping = [
    Row(file_id='F001', file_name='ds_assessment_jan.csv',
        course='Data Science', period='2024-01', target_table='bronze.ds_raw',
        active=True, expected_rows=4),
    Row(file_id='F002', file_name='eng_assessment_jan.csv',
        course='Engineering', period='2024-01', target_table='bronze.eng_raw',
        active=True, expected_rows=3),
    Row(file_id='F003', file_name='maths_assessment_jan.csv',
        course='Mathematics', period='2024-01', target_table='bronze.maths_raw',
        active=True, expected_rows=3),
    Row(file_id='F004', file_name='ds_assessment_feb.csv',
        course='Data Science', period='2024-02', target_table='bronze.ds_raw_feb',
        active=False, expected_rows=4),  # inactive — will be skipped
]

df_map = spark.createDataFrame(file_mapping)
df_map.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.config.file_mapping')
print('File mapping table created:')
df_map.show()

---
## Step 1 — generate_file_list
---

In [ ]:
# ── TASK 1: generate_file_list ───────────────────────────────────────────────
# Reads the mapping table and builds the For-Each input array.
# Only includes ACTIVE files for the current period.
import json, datetime

current_period = '2024-01'  # In production: calculate from job run date

df_map = spark.table('demo_catalog.config.file_mapping')

# Filter to active files for current period only
active_files = df_map.filter(
    (df_map.active == True) & (df_map.period == current_period)
)

file_list = [
    {
        'file_id':       r['file_id'],
        'file_name':     r['file_name'],
        'course':        r['course'],
        'period':        r['period'],
        'target_table':  r['target_table'],
        'expected_rows': r['expected_rows']
    }
    for r in active_files.collect()
]

print(f'Files to process for period {current_period}: {len(file_list)}')
for f in file_list:
    print(f'  {f["file_id"]} — {f["file_name"]} -> {f["target_table"]}')

dbutils.jobs.taskValues.set(key='file_list',      value=json.dumps(file_list))
dbutils.jobs.taskValues.set(key='period',         value=current_period)
dbutils.jobs.taskValues.set(key='files_expected', value=int(len(file_list)))
print(f'file_list published. For-Each will create {len(file_list)} parallel iterations.')

---
## Step 2 — ingest_single_file (nested For-Each task)
---

### For-Each configuration in Job UI
```
Inputs:      {{tasks.generate_file_list.values.file_list}}
Concurrency: 10  (process 10 files simultaneously)

Nested task parameters:
  input  =  {{input}}
```

In [ ]:
# ── NESTED TASK: ingest_single_file ─────────────────────────────────────────
# Runs once per file in file_list.
# Receives the full file config dict via {{input}} widget parameter.
import json
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp
import datetime, random

try:
    raw = dbutils.widgets.get('input')
    params = json.loads(raw)
except:
    params = {'file_id': 'F001', 'file_name': 'ds_assessment_jan.csv',
              'course': 'Data Science', 'period': '2024-01',
              'target_table': 'bronze.ds_raw', 'expected_rows': 4}

file_id      = params['file_id']
file_name    = params['file_name']
course       = params['course']
period       = params['period']
target_table = params['target_table']
expected     = params['expected_rows']

print(f'[{file_id}] Ingesting: {file_name}')
print(f'  Course: {course}  |  Period: {period}  |  Target: {target_table}')

# Simulate reading from a file (in production: spark.read.csv(file_path))
# We generate dummy data matching the course
random.seed(hash(file_id))

if course == 'Data Science':
    records = [
        Row(student_id='S001', name='Aisha Rahman',  score=82, attendance=95, course=course, period=period),
        Row(student_id='S002', name='James Bradley', score=41, attendance=60, course=course, period=period),
        Row(student_id='S007', name='Liu Wei',       score=67, attendance=80, course=course, period=period),
        Row(student_id='S009', name='Tariq Ahmed',   score=29, attendance=40, course=course, period=period),
    ]
elif course == 'Engineering':
    records = [
        Row(student_id='S003', name='Priya Patel',  score=76, attendance=88, course=course, period=period),
        Row(student_id='S004', name='Carlos Mendez',score=55, attendance=72, course=course, period=period),
        Row(student_id='S008', name='Emma Johnson', score=74, attendance=85, course=course, period=period),
    ]
else:  # Mathematics
    records = [
        Row(student_id='S005', name='Sophie Williams', score=91, attendance=98, course=course, period=period),
        Row(student_id='S006', name='Kwame Asante',    score=38, attendance=55, course=course, period=period),
        Row(student_id='S010', name='Fatima Al-Sayed', score=88, attendance=94, course=course, period=period),
    ]

df = (spark.createDataFrame(records)
        .withColumn('source_file', lit(file_name))
        .withColumn('ingested_at', current_timestamp()))

# Write to the target table specified in the config
full_table = f'demo_catalog.{target_table}'
df.write.format('delta').mode('overwrite').saveAsTable(full_table)

actual_rows = df.count()
row_match   = actual_rows == expected

print(f'  Written {actual_rows} rows to {full_table}')
print(f'  Expected: {expected}  |  Actual: {actual_rows}  |  Match: {row_match}')

---
## Step 3 — post-ingestion audit
---

In [ ]:
# ── TASK 3: post_ingest_audit ────────────────────────────────────────────────
# Runs after the For-Each completes. Reads all three tables and summarises.
import json

try:
    files_expected = int(dbutils.widgets.get('files_expected'))
    period         = dbutils.widgets.get('period')
except:
    files_expected, period = 3, '2024-01'

print(f'Post-ingestion audit for period: {period}')
print(f'Files expected: {files_expected}')
print()

# Check all target tables
targets = [
    ('demo_catalog.bronze.ds_raw',    'Data Science'),
    ('demo_catalog.bronze.eng_raw',   'Engineering'),
    ('demo_catalog.bronze.maths_raw', 'Mathematics'),
]

total_rows = 0
print(f'{"Table":<40} {"Rows":>6}')
print('-' * 48)
for table, course in targets:
    try:
        count = spark.table(table).count()
        total_rows += count
        print(f'{table:<40} {count:>6}')
    except:
        print(f'{table:<40}  NOT FOUND')

print('-' * 48)
print(f'{"TOTAL":<40} {total_rows:>6}')
print()
print(f'All {files_expected} files ingested successfully for period {period}.')